# Module 09 · Subsetting and state annotation

Take one cell type, rebuild its manifold, and assign each subcluster to an
activation state.

Everything upstream worked at cell-type level. From here the analysis is
*within* a cell type, on a manifold computed from that cell type alone —
because the variation that separates microglial states is invisible in a
PCA dominated by neurons.

**The object.** `mg <- obj_ct` is the same cells with no re-filtering:
renormalized, re-HVG'd at 3000, rescaled regressing `nCount_RNA`, re-Harmony'd
at `theta = 2`, with its own `umap.mg` and resolution-0.6 clusters. `mg` is
canonical from here; `obj_ct` is only the input.

**States are assigned at cluster level, not cell level.** Each subcluster gets
the state whose z-scored panel mean is highest. Cell-level argmax would assign
a state to every cell whether or not the population supports one; the cluster
is the unit that has enough cells to make the call meaningful.

| Section | |
|---|---|
| 01-03 | config, load, preflight |
| 04-05 | score the five panels, annotate by cluster-level z-argmax |
| 06 | state UMAP |
| 07-08 | composition, Garg differential proportion, state-level burden GLMM |
| 09-10 | subtypes as % of all cells, is the selected state higher in disease |

Nothing here is specific to one state — all five panels are scored and all five
are annotated. `AXIS` only decides which one later modules focus on.

---
## 01 · Config

**Why.** Self-contained — the notebook carries its own config rather than
importing one, so it can be read and run without tracing an import elsewhere.
Paths come from the environment; see `.env.example`.

**`AXIS` is the one thing you change.** Column names, quadrant labels, figure
titles, contrast names and output directories all derive from it. Outputs are
namespaced by axis so two runs never overwrite each other.

```
AXIS <- "IRM"    # "IRM" | "DAM_like" | "ARM" | "Stress"
```

`STATE_ORDER` and `STATE_COLORS` list all five states regardless of `AXIS` —
those are the annotation registry, not a per-run choice.

In [ ]:
# =============================================================================
# CONFIG
# =============================================================================
# Paths come from the environment - see .env.example. Nothing below hardcodes
# a filesystem location.
#   SENESCENCE_DATA : analysis root (module outputs written under it)
#   SENESCENCE_REF  : reference root (published panels, read-only)
# =============================================================================

SCRATCH <- Sys.getenv("SENESCENCE_DATA")
REF_DIR <- Sys.getenv("SENESCENCE_REF")
if (SCRATCH == "" || REF_DIR == "")
    stop("SENESCENCE_DATA and SENESCENCE_REF must be set. See .env.example.")


# ─────────────────────────────────────────────────────────────────────────────
# Libraries
# ─────────────────────────────────────────────────────────────────────────────
suppressPackageStartupMessages({
    library(Seurat)
    library(Matrix)
    library(dplyr)
    library(tidyr)
    library(readxl)
    library(ggplot2)
    library(patchwork)
    library(scales)
    library(qs)
    library(jsonlite)
    library(lme4)
    library(lmerTest)
    library(MASS)
    library(robustbase)
    library(broom)
    library(broom.mixed)
})

# Fix MASS::select masking dplyr::select
select <- dplyr::select


# ─────────────────────────────────────────────────────────────────────────────
# Inline plotting viewport (Jupyter / IRkernel)
# ─────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────
# Run parameters — edit these
# ─────────────────────────────────────────────────────────────────────────────
TISSUE     <- "brain"
STUDY_TYPE <- "disease"
DISEASE    <- "AD"
DATASET    <- "psychad_ad"

CELL_TYPE  <- "Microglia"


# ─────────────────────────────────────────────────────────────────────────────
# Stratification
# ─────────────────────────────────────────────────────────────────────────────
STRATIFY_BY_GROUP     <- TRUE
STRATIFICATION_GROUPS <- c("Old_AD", "Old_Healthy_Control")


# ─────────────────────────────────────────────────────────────────────────────
# Statistical parameters
# ─────────────────────────────────────────────────────────────────────────────
STATISTICAL_PARAMS <- list(
    min_cells_per_group  = 5L,
    fdr_threshold        = 0.05,
    fdr_method           = "BH",
    bootstrap_n_iter     = 100L,
    bootstrap_seed       = 42L,
    seed                 = 42L,
    confidence_level     = 0.95,
    rationale            = "M09-equivalent thresholds with M05 organizational rewrite"
)

set.seed(STATISTICAL_PARAMS$seed)


# ─────────────────────────────────────────────────────────────────────────────
# Derived condition tags
# ─────────────────────────────────────────────────────────────────────────────
IS_AGING          <- (STUDY_TYPE == "aging")
IS_DISEASE        <- (STUDY_TYPE == "disease")

BASE_M04   <- file.path(SCRATCH, TISSUE, "module_04T_tissue_export",
                        CONDITION_SUBPATH, DATASET)
BASE_M05   <- file.path(SCRATCH, TISSUE, "module_05_senescence_enrichment",
                        CONDITION_SUBPATH, DATASET, CELL_TYPE)

PATHS <- list(
    m04_root       = BASE_M04,
    m04_seurat     = file.path(BASE_M04, paste0(DATASET, "_tissue_seurat.qs")),
    m04_manifest   = file.path(BASE_M04, "manifest.json"),
    output_root    = BASE_M05,
    data           = file.path(BASE_M05, "data"),
    results        = file.path(BASE_M05, "results"),
    figures        = file.path(BASE_M05, "figures"),
    logs           = file.path(BASE_M05, "_logs"),
    scored_qs      = file.path(BASE_M05, "data",
                               paste0(CELL_TYPE, "_scored.qs")),
    gene_lists_rds = file.path(BASE_M05, "data", "gene_lists.rds"),
    manifest       = file.path(BASE_M05, "_logs", "m05_manifest.json"),
    markers_dir    = file.path(REF_DIR, "markers"),
    sloan_xlsx     = file.path(REF_DIR, "markers", "1-s2.0-S2666979X25003830-mmc10.xlsx"),
    senmayo_xlsx   = file.path(REF_DIR, "markers", "41467_2022_32552_MOESM4_ESM.xlsx"),
    fridman_gmt    = file.path(REF_DIR, "markers", "FRIDMAN_SENESCENCE_UP.v2026.1.Hs.gmt")
)

for (key in c("data", "results", "figures", "logs")) {
    dir.create(PATHS[[key]], recursive = TRUE, showWarnings = FALSE)
}


# ─────────────────────────────────────────────────────────────────────────────
# Color palettes
# ─────────────────────────────────────────────────────────────────────────────
LINEAGE_COLORS_BY_TISSUE <- list(
    brain = c(
        Excitatory      = "#0072B2",
        Inhibitory      = "#E69F00",
        Astrocyte       = "#009E73",
        Oligodendrocyte = "#56B4E9",
        Microglia       = "#D55E00",
        OPC             = "#CC79A7",
        Endothelial     = "#7F7F7F",
        Pericyte        = "#999999",
        VLMC            = "#A9A9A9",
        VSMC            = "#696969",
        PVM             = "#FF6347",
        Adaptive        = "#FFD700"
    ),
    pbmc = c(
        cd4t = "#4E79A7", cd8t = "#A0CBE8", unconvT = "#BAB0AC",
        nkcell = "#59A14F", cd14mono = "#F28E2B", cd16mono = "#FFBE7D",
        memB = "#B07AA1", naiveB = "#76B7B2", dc = "#9C755F"
    ),
    csf = c(
        cd4t = "#4E79A7", cd8t = "#A0CBE8", nkcell = "#59A14F",
        monocyte = "#F28E2B", bcell = "#B07AA1", dc = "#76B7B2"
    )
)
LINEAGE_COLORS <- LINEAGE_COLORS_BY_TISSUE[[TISSUE]]

SNC_COLORS <- c(
    Senescent       = "#C44E52",
    `Non-senescent` = "#D3D3D3",
    `TRUE`          = "#C44E52",
    `FALSE`         = "#D3D3D3",
    True            = "#C44E52",
    False           = "#D3D3D3"
)

STUDY_GROUP_COLORS <- c(
    Age_20_29 = "#2E86AB", Age_30_39 = "#4A90E2", Age_40_49 = "#50C878",
    Age_50_59 = "#FFB347", Age_60_69 = "#FF8C00", Age_70_79 = "#E24A4A",
    Age_80_100 = "#8B0000",
    Control = "#4E79A7", MCI = "#F28E2B", AD = "#E15759",
    Young_Healthy_Control = "#4A90E2",
    Old_Healthy_Control   = "#4E79A7",
    Old_AD                = "#E15759",
    All                   = "#7F7F7F"
)

PHASE_COLORS <- c(G1 = "#4E79A7", S = "#F28E2B", G2M = "#E15759")

SEX_COLORS <- c(
    Male = "#5D6D7E", Female = "#A569BD",
    M    = "#5D6D7E", F      = "#A569BD"
)

MODEL_AGREEMENT_COLORS <- c(
    `Up (sig)`     = "#C44E52",
    `Up (ns)`      = "#F4B5B5",
    `ns`           = "#D3D3D3",
    `Down (ns)`    = "#A8C5DC",
    `Down (sig)`   = "#3B6F8F"
)


# ─────────────────────────────────────────────────────────────────────────────
# Plot style
# ─────────────────────────────────────────────────────────────────────────────
PLOT_STYLE <- list(
    dpi        = 150,
    dpi_save   = 300,
    font_size  = 10,
    title_size = 11,
    formats    = c("pdf", "png", "svg"),
    pt_size    = 0.05,
    label_size = 4
)

theme_clean <- function(base_size = PLOT_STYLE$font_size) {
    theme_classic(base_size = base_size) +
    theme(
        plot.title       = element_text(size = PLOT_STYLE$title_size,
                                        face = "plain", hjust = 0),
        legend.title     = element_text(size = base_size, face = "plain"),
        panel.border     = element_rect(color = "black", fill = NA, linewidth = 0.5),
        panel.grid       = element_blank(),
        axis.line        = element_blank()
    )
}


# ─────────────────────────────────────────────────────────────────────────────
# Formatting helpers
# ─────────────────────────────────────────────────────────────────────────────
fmt_n <- function(n) format(round(n), big.mark = ",", scientific = FALSE)

fmt_size <- function(path) {
    if (!file.exists(path)) return("missing")
    sz <- file.size(path)
    if (sz > 1024^3) return(sprintf("%.2f GB", sz / 1024^3))
    if (sz > 1024^2) return(sprintf("%.1f MB", sz / 1024^2))
    sprintf("%.1f KB", sz / 1024)
}

fmt_pct <- function(num, denom) {
    if (denom == 0) return(sprintf("%s (--)", fmt_n(num)))
    sprintf("%s (%.1f%%)", fmt_n(num), num / denom * 100)
}

fmt_elapsed <- function(secs) {
    if (secs < 60)   return(sprintf("%.1f sec", secs))
    if (secs < 3600) return(sprintf("%.1f min", secs / 60))
    sprintf("%.1f hr", secs / 3600)
}

fmt_p <- function(p) {
    if (is.na(p)) return("NA")
    if (p < 0.001) return(sprintf("%.2e", p))
    sprintf("%.3f", p)
}

fmt_p_short <- function(p) {
    if (is.na(p)) return("--")
    if (p < 0.001) return(sprintf("%.1e", p))
    sprintf("%.3f", p)
}

sig_stars <- function(p) {
    ifelse(is.na(p), "",
    ifelse(p < 0.001, "***",
    ifelse(p < 0.01,  "**",
    ifelse(p < 0.05,  "*", "ns"))))
}

now_iso <- function() format(Sys.time(), "%Y-%m-%dT%H:%M:%S")

bytes_str <- function(x) format(x, scientific = FALSE, trim = TRUE)


# ─────────────────────────────────────────────────────────────────────────────
# Color helpers
# ─────────────────────────────────────────────────────────────────────────────
text_color_for_bg <- function(hex) {
    rgb_vals  <- col2rgb(hex)
    luminance <- 0.299 * rgb_vals[1, ] + 0.587 * rgb_vals[2, ] + 0.114 * rgb_vals[3, ]
    ifelse(luminance < 140, "white", "black")
}


# ─────────────────────────────────────────────────────────────────────────────
# Time / save helpers
# ─────────────────────────────────────────────────────────────────────────────
time_step <- function(label, expr) {
    cat(sprintf("\n▸ %s\n", label))
    t0  <- Sys.time()
    res <- expr
    elapsed <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
    cat(sprintf("  ✓ %s  (%s)\n", label, fmt_elapsed(elapsed)))
    res
}

save_figure <- function(fig, slug, width = 10, height = 7) {
    for (ext in PLOT_STYLE$formats) {
        path <- file.path(PATHS$figures, paste0(slug, ".", ext))
        ggsave(path, fig, width = width, height = height,
               dpi = PLOT_STYLE$dpi_save, bg = "white")
    }
    cat(sprintf("  ✓ saved → figures/%s.{%s}\n",
                slug, paste(PLOT_STYLE$formats, collapse = ",")))
}

save_table <- function(df, slug, row.names = FALSE) {
    path <- file.path(PATHS$results, paste0(slug, ".csv"))
    write.csv(df, path, row.names = row.names)
    cat(sprintf("  ✓ saved → results/%s.csv  (%d rows)\n",
                slug, nrow(df)))
}


# ─────────────────────────────────────────────────────────────────────────────
# filter_to_stratum() — slice metadata to one stratum
# ─────────────────────────────────────────────────────────────────────────────
filter_to_stratum <- function(md, stratum, study_group_col) {
    if (stratum == "All") return(md)
    md[md[[study_group_col]] == stratum, , drop = FALSE]
}


# ─────────────────────────────────────────────────────────────────────────────
# tidy_model_results() — standardize one-row result records across all models
# ─────────────────────────────────────────────────────────────────────────────
tidy_model_results <- function(stratum, outcome, model,
                               n_donors, n_cells_test, n_cells_ref,
                               estimate, se, ci_low, ci_high,
                               statistic, p_value,
                               extra = NULL) {
    out <- data.frame(
        stratum      = as.character(stratum),
        outcome      = as.character(outcome),
        model        = as.character(model),
        n_donors     = as.integer(n_donors),
        n_cells_test = as.integer(n_cells_test),
        n_cells_ref  = as.integer(n_cells_ref),
        estimate     = as.numeric(estimate),
        se           = as.numeric(se),
        ci_low       = as.numeric(ci_low),
        ci_high      = as.numeric(ci_high),
        statistic    = as.numeric(statistic),
        p_value      = as.numeric(p_value),
        stringsAsFactors = FALSE
    )
    if (!is.null(extra) && length(extra) > 0) {
        for (nm in names(extra)) {
            v <- extra[[nm]]
            if (length(v) != 1) v <- I(list(v))
            out[[nm]] <- v
        }
    }
    out
}


# ─────────────────────────────────────────────────────────────────────────────
# Library version log
# ─────────────────────────────────────────────────────────────────────────────
R_PKG_VERSIONS <- list(
    R          = R.version$version.string,
    Seurat     = as.character(packageVersion("Seurat")),
    Matrix     = as.character(packageVersion("Matrix")),
    dplyr      = as.character(packageVersion("dplyr")),
    tidyr      = as.character(packageVersion("tidyr")),
    readxl     = as.character(packageVersion("readxl")),
    ggplot2    = as.character(packageVersion("ggplot2")),
    patchwork  = as.character(packageVersion("patchwork")),
    qs         = as.character(packageVersion("qs")),
    jsonlite   = as.character(packageVersion("jsonlite")),
    lme4       = as.character(packageVersion("lme4")),
    lmerTest   = as.character(packageVersion("lmerTest")),
    MASS       = as.character(packageVersion("MASS")),
    robustbase = as.character(packageVersion("robustbase")),
    broom      = as.character(packageVersion("broom")),
    broom.mixed = as.character(packageVersion("broom.mixed"))
)



# =============================================================================
# §0.2 — AXIS SELECT
# =============================================================================
# The ONE thing you change to re-run the whole flow on a different state.
# Everything downstream — column names, quadrant labels, figure titles,
# legends, output paths — derives from this. Nothing is hardcoded per state.
# =============================================================================

AXIS <- "IRM"          # <<< "IRM" | "DAM_like" | "ARM" | "Stress"

# --- registry: score column candidates + display label + canonical hex -------
# Score columns are looked up in order; the first present on the object wins.
AXIS_REGISTRY <- list(
    # tag = short token used in CONTRAST NAMES and therefore in GSEA/DE filenames.
    # It is deliberately NOT the same as `lab` (display) or the list key: existing
    # results on disk are named DAMaxis_*, not DAM_likeaxis_*.
    IRM      = list(cols = c("Score_IRM",      "IRM1"),      lab = "IRM",      tag = "IRM",    hex = "#2980B9"),
    DAM_like = list(cols = c("Score_DAM_like", "DAM_like1"), lab = "DAM-like", tag = "DAM",    hex = "#C0392B"),
    ARM      = list(cols = c("Score_ARM",      "ARM1"),      lab = "ARM",      tag = "ARM",    hex = "#E67E22"),
    Stress   = list(cols = c("Score_Stress",   "Stress1"),   lab = "Stress",   tag = "Stress", hex = "#8E44AD")
)
stopifnot(AXIS %in% names(AXIS_REGISTRY))

AXIS_SPEC <- AXIS_REGISTRY[[AXIS]]
X_LAB     <- AXIS_SPEC$lab
X_HEX     <- AXIS_SPEC$hex
Y_LAB     <- "Senescence"
Y_TAG     <- "SnC"
X_TAG     <- AXIS_SPEC$tag
SEN_CANDIDATES <- c("senescence_score", "SenePy_score")

# --- quadrant labels derive from the axis -----------------------------------
QUAD_COL    <- paste0("quad4_", tolower(AXIS))
QUAD_LEVELS <- c(sprintf("Sen- %s-", X_LAB), sprintf("Sen+ %s-", X_LAB),
                 sprintf("Sen- %s+", X_LAB), sprintf("Sen+ %s+", X_LAB))
QUAD_COLORS <- setNames(c("#B8B8B8", "#2E7D32", X_HEX, "#6A1B9A"), QUAD_LEVELS)

# --- the four DE / GSEA contrasts, named off the tags ------------------------
CONTRASTS <- c(sprintf("%saxis_%spos", X_TAG, Y_TAG),   # axis effect within SnC+
               sprintf("%saxis_%sneg", X_TAG, Y_TAG),   # axis effect within SnC-
               sprintf("%saxis_%spos", Y_TAG, X_TAG),   # sen effect within axis+
               sprintf("%saxis_%sneg", Y_TAG, X_TAG))   # sen effect within axis-
CONTRAST_HEADERS <- c(
    sprintf("%s+%s+ vs %s+%s−", Y_TAG, X_LAB, Y_TAG, X_LAB),
    sprintf("%s−%s+ vs %s−%s−", Y_TAG, X_LAB, Y_TAG, X_LAB),
    sprintf("%s+%s+ vs %s−%s+", Y_TAG, X_LAB, Y_TAG, X_LAB),
    sprintf("%s+%s− vs %s−%s−", Y_TAG, X_LAB, Y_TAG, X_LAB))

# --- outputs are namespaced by axis so runs never overwrite each other -------
AXIS_FIG_DIR <- file.path(PATHS$figures, AXIS)
AXIS_RES_DIR <- file.path(PATHS$results, AXIS)
dir.create(AXIS_FIG_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(AXIS_RES_DIR, recursive = TRUE, showWarnings = FALSE)

# --- state annotation (all five; NOT gated by AXIS) --------------------------
STATE_ORDER  <- c("Homeostatic", "ARM", "IRM", "Stress", "DAM_like")
STATE_COLORS <- c(Homeostatic = "#7F8C8D", ARM = "#E67E22", IRM = "#2980B9",
                  Stress = "#8E44AD", DAM_like = "#C0392B")

# --- DE model design (confirmed 2026-07-28; supersedes the leaner ~pop+grp2+Sex)
DE_DESIGN     <- "~ 0 + pop + grp2 + Sex + Mean_Log_Library_Depth_scaled + Cohort"
DE_COEF       <- "popTEST"
DE_MIN_CELLS  <- 10L

# --- GSEA (consumed by the python notebook via gsea_config.json) -------------
GSEA_DBS      <- c("Reactome_2022")
GSEA_FDR_SIG  <- 0.05
GSEA_N_COMMON <- 6L      # sig in >=3 contrasts, top N by mean NES
GSEA_N_UNIQUE <- 4L      # sig in exactly 1 contrast, top N by |NES|
GSEA_RIBO_STRIP <- FALSE # keep translational terms; see Part F Why



# §0.3 — AXIS-AWARE HELPERS  (one definition each — see note)
# =============================================================================
# In the source notebook fit_one was defined 13x, resolve_score_col 6x,
# z_score 5x, venn2 4x, and save_figure was REDEFINED at cells 317/341,
# shadowing the canonical version above. Everything lives here now so a
# later cell cannot silently shadow it.
# =============================================================================

# --- resolve a score column from candidates ---------------------------------
resolve_score_col <- function(md, candidates, what = "score") {
    hit <- candidates[candidates %in% colnames(md)]
    if (!length(hit)) stop(sprintf("no %s column found; tried: %s",
                                   what, paste(candidates, collapse = ", ")))
    hit[1]
}

z_score <- function(x) as.numeric(scale(x))

# --- build the SnC x AXIS quadrant column -----------------------------------
# Mean-split on z-scored values, exactly as the source (cell 54).
build_quadrants <- function(obj, axis = AXIS, verbose = TRUE) {
    md   <- obj@meta.data
    spec <- AXIS_REGISTRY[[axis]]
    sen  <- resolve_score_col(md, SEN_CANDIDATES, "senescence")
    xcol <- resolve_score_col(md, spec$cols, paste(axis, "score"))
    sz <- z_score(md[[sen]]); xz <- z_score(md[[xcol]])
    lab <- spec$lab
    q <- ifelse(sz >  0 & xz >  0, sprintf("Sen+ %s+", lab),
        ifelse(sz >  0 & xz <= 0, sprintf("Sen+ %s-", lab),
        ifelse(sz <= 0 & xz >  0, sprintf("Sen- %s+", lab),
                                  sprintf("Sen- %s-", lab))))
    q[is.na(sz) | is.na(xz)] <- NA
    obj[[paste0("quad4_", tolower(axis))]] <- q
    obj$sen_z <- sz
    obj$axis_z <- xz
    if (verbose) {
        cat(sprintf("  scores : sen=%s  %s=%s\n", sen, axis, xcol))
        print(table(q, useNA = "ifany"))
        cat(sprintf("  cor(sen_z, %s_z) = %.3f   <- independence check\n",
                    tolower(axis), cor(sz, xz, use = "complete.obs")))
    }
    obj
}

# --- PREFLIGHT: fail loudly before any model runs ---------------------------
preflight_axis <- function(obj, axis = AXIS, min_cells = 50L, donor_col = "Donor") {
    md <- obj@meta.data; ok <- TRUE
    say <- function(pass, msg) {
        cat(sprintf("  [%s] %s\n", if (pass) "OK  " else "FAIL", msg))
        if (!pass) ok <<- FALSE
    }
    cat(sprintf("\n── PREFLIGHT · axis = %s ──\n", axis))
    say(axis %in% names(AXIS_REGISTRY), sprintf("axis '%s' is registered", axis))
    spec <- AXIS_REGISTRY[[axis]]
    xhit <- spec$cols[spec$cols %in% colnames(md)]
    say(length(xhit) > 0, sprintf("score column present (%s)",
        if (length(xhit)) xhit[1] else paste(spec$cols, collapse = "/")))
    shit <- SEN_CANDIDATES[SEN_CANDIDATES %in% colnames(md)]
    say(length(shit) > 0, "senescence score column present")
    if (length(xhit) && length(shit)) {
        say(sd(md[[xhit[1]]], na.rm = TRUE) > 0, "axis score is non-constant")
        say(sd(md[[shit[1]]], na.rm = TRUE) > 0, "senescence score is non-constant")
    }
    qc <- paste0("quad4_", tolower(axis))
    if (qc %in% colnames(md)) {
        tb <- table(md[[qc]])
        say(length(tb) == 4, sprintf("all four quadrants populated (%d)", length(tb)))
        say(all(tb >= min_cells), sprintf("every quadrant >= %d cells (min %d)",
                                          min_cells, min(tb)))
        if (donor_col %in% colnames(md)) {
            nd <- tapply(md[[donor_col]], md[[qc]], function(z) length(unique(z)))
            say(all(nd >= 2), sprintf("every quadrant has >=2 donors (min %d)", min(nd)))
        }
    } else cat(sprintf("  [--  ] %s not built yet (run B1)\n", qc))
    cat(sprintf("── %s ──\n\n", if (ok) "PASS" else "STOP: fix before proceeding"))
    invisible(ok)
}

# --- ONE mixed-model fitter (replaces 13 copies of fit_one) -----------------
# formula_str is built by the caller, so every Part C analysis is this
# function with a different formula and a different subset.
fit_lmm <- function(df, formula_str, term, label = NA_character_) {
    fit <- tryCatch(lmerTest::lmer(as.formula(formula_str), data = df,
                                   REML = TRUE,
                                   control = lme4::lmerControl(
                                       optimizer = "bobyqa",
                                       optCtrl = list(maxfun = 2e5))),
                    error = function(e) NULL, warning = function(w) NULL)
    if (is.null(fit)) return(data.frame(label = label, term = term,
                                        beta = NA, se = NA, ci_low = NA,
                                        ci_high = NA, p_value = NA,
                                        n = nrow(df), converged = FALSE))
    co <- summary(fit)$coefficients
    if (!term %in% rownames(co)) return(data.frame(label = label, term = term,
                                        beta = NA, se = NA, ci_low = NA,
                                        ci_high = NA, p_value = NA,
                                        n = nrow(df), converged = FALSE))
    b <- co[term, "Estimate"]; s <- co[term, "Std. Error"]
    data.frame(label = label, term = term, beta = b, se = s,
               ci_low = b - 1.96 * s, ci_high = b + 1.96 * s,
               p_value = co[term, "Pr(>|t|)"], n = nrow(df), converged = TRUE)
}

# --- ONE forest renderer (replaces 7 near-copies) ---------------------------
forest_plot <- function(res, title = "", xlab = "beta (95% CI)",
                        facet = NULL, color = X_HEX) {
    stopifnot(all(c("label", "beta", "ci_low", "ci_high") %in% names(res)))
    if (!"p_adj" %in% names(res))
        res$p_adj <- p.adjust(res$p_value, method = STATISTICAL_PARAMS$fdr_method)
    res$sig <- sig_stars(res$p_adj)
    res$label <- factor(res$label, levels = rev(unique(res$label)))
    p <- ggplot(res, aes(x = beta, y = label)) +
        geom_vline(xintercept = 0, linetype = "dashed",
                   colour = "grey60", linewidth = 0.3) +
        geom_errorbarh(aes(xmin = ci_low, xmax = ci_high),
                       height = 0, linewidth = 0.4, colour = color) +
        geom_point(size = 1.8, colour = color) +
        geom_text(aes(x = ci_high, label = sig), hjust = -0.35,
                  size = 2.6, na.rm = TRUE) +
        labs(title = title, x = xlab, y = NULL) +
        theme_clean() +
        theme(panel.grid.major.y = element_line(colour = "grey92", linewidth = 0.25))
    if (!is.null(facet)) p <- p + facet_wrap(as.formula(paste("~", facet)), scales = "free_x")
    p + coord_cartesian(clip = "off")
}

# --- block banner: prints the Why with the axis resolved --------------------
say_block <- function(id, title, why = NULL) {
    cat("\n", strrep("═", 76), "\n", sep = "")
    cat(sprintf("%s  ·  %s\n", id, sprintf(title, X_LAB)))
    cat(strrep("═", 76), "\n", sep = "")
    if (!is.null(why)) cat(sprintf("WHY: %s\n\n", sprintf(why, X_LAB)))
}

# X_COL is resolved in the load section, once the object exists:
#     X_COL <- resolve_score_col(mg@meta.data, AXIS_SPEC$cols, 'axis score')
# Every downstream cell reads X_COL, never a literal score column name.

---
## 02 · Load and derive the subset manifold

**Why.** The subset needs its own embedding. Normalizing, selecting variable
features and running PCA on the parent object bakes in the parent's dominant
axes; redoing it on the subset lets within-cell-type structure surface.

**No re-filtering.** Identical cells in, identical cells out — only the
representation changes. That matters because every count downstream (state
composition, burden per state) has to reconcile with the module 07 totals.

**Steps.** `NormalizeData` → `FindVariableFeatures` (3000) → `ScaleData`
regressing `nCount_RNA` → `RunPCA` → `RunHarmony` (theta = 2) → `RunUMAP`
(`umap.mg`) → `FindClusters` (resolution 0.6).

**Resolve the axis column here.** Once `mg` exists:
`X_COL <- resolve_score_col(mg@meta.data, AXIS_SPEC$cols, 'axis score')`.

In [ ]:
# STEP 1 — microglia subclustering on a COPY (mg). theta=2, no re-filtering.
#   obj_ct stays untouched. Same cell set, microglia-specific embedding.
suppressPackageStartupMessages({ library(Seurat); library(harmony) })

N_VARIABLE_FEATURES <- 3000
N_PCS               <- 50
N_DIMS_USE          <- 15
CLUSTERING_RESOLUTION <- 0.6
BATCH_COL           <- "Cohort"
HARMONY_THETA       <- 2

mg <- obj_ct                      # work on a copy; obj_ct preserved
DefaultAssay(mg) <- "RNA"
cat(sprintf("mg: %d cells (same as obj_ct, no QC re-filter)\n", ncol(mg)))

# reset to raw counts processing (re-derive everything microglia-specific)
cat("1. Normalize\n");        mg <- NormalizeData(mg, verbose=FALSE)
cat("2. Variable features\n");mg <- FindVariableFeatures(mg, nfeatures=N_VARIABLE_FEATURES, verbose=FALSE)
cat("3. Scale (regress nCount_RNA)\n")
mg <- ScaleData(mg, features=VariableFeatures(mg), vars.to.regress="nCount_RNA", verbose=FALSE)
cat("4. PCA\n");              mg <- RunPCA(mg, npcs=N_PCS, verbose=FALSE)

cat(sprintf("5. Harmony (batch=%s, theta=%d)\n", BATCH_COL, HARMONY_THETA))
mg <- RunHarmony(mg, group.by.vars=BATCH_COL, theta=c(HARMONY_THETA),
                 max_iter=30, verbose=FALSE)

cat("6. UMAP (harmony dims 1:15) → stored as 'umap.mg'\n")
mg <- RunUMAP(mg, reduction="harmony", dims=1:N_DIMS_USE,
              reduction.name="umap.mg", reduction.key="umapmg_", verbose=FALSE)

cat("7. Neighbors + cluster (res=", CLUSTERING_RESOLUTION, ")\n")
mg <- FindNeighbors(mg, reduction="harmony", dims=1:N_DIMS_USE, verbose=FALSE)
mg <- FindClusters(mg, resolution=CLUSTERING_RESOLUTION, verbose=FALSE)

mg$mg_cluster <- Idents(mg)       # explicit, non-colliding name
cat(sprintf("\n✓ %d microglia subclusters\n", length(unique(mg$mg_cluster))))
cat("Cells per subcluster:\n"); print(table(mg$mg_cluster))

---
## 03 · Preflight

**Why.** Fails loudly before any model runs, rather than producing a figure
that is quietly about the wrong state. Checks that the axis is in the registry,
that its score column is present on the object, and that enough cells and
donors survive to fit anything.

In [ ]:
preflight_axis(mg)

---
## 04 · Score the five state panels

**Why.** One module score per canonical microglial state, from published
panels. All five are scored on every run — the annotation in section 05 needs
the full set to take an argmax over, and restricting to `AXIS` would make the
assignment depend on which state you happened to be looking at.

**Panels.** DAM-like (25 genes) · ARM (18) · IRM (19) · Homeostatic (15) ·
Stress (22).

**Test.** `AddModuleScore` on log-normalized expression, `ctrl = min(100,
n_genes)`, seed 42 — the same machinery as the Sloan modules in module 05, so
the scores are on a comparable scale.

**Display.** Per-panel gene-detection counts. A panel below 5 detected genes is
skipped rather than scored on a fragment.

In [ ]:
# STEP 2 — score the 5 microglia state panels on mg (→ ...1 columns)
suppressPackageStartupMessages({ library(Seurat) })

gene_sets <- MICROGLIA_STATE_PANELS     # alias the annotation module expects
DefaultAssay(mg) <- "RNA"

cat("Scoring panels on mg subclustered object:\n")
for (st in names(gene_sets)) {
    genes <- intersect(toupper(gene_sets[[st]]), toupper(rownames(mg)))
    # match case to actual rownames
    genes <- rownames(mg)[toupper(rownames(mg)) %in% toupper(gene_sets[[st]])]
    mg <- AddModuleScore(mg, features=list(genes), name=st, seed=42)
    # AddModuleScore appends "1" → column is e.g. "DAM_like1"
    cat(sprintf("  %-14s %2d/%2d genes found → %s1\n",
                st, length(genes), length(gene_sets[[st]]), st))
}

# verify the expected columns exist
score_cols <- paste0(names(gene_sets), "1")
cat("\nScore columns present:",
    all(score_cols %in% colnames(mg@meta.data)), "\n")
print(score_cols)

---
## 05 · Annotate states — cluster-level z-argmax

**Why.** The assignment unit is the subcluster. Within each cluster, take the
mean of each state score, z-score those means across clusters, and give the
cluster the state with the highest z.

Z-scoring across clusters before the argmax is what makes the panels
comparable — raw module scores differ in scale between panels because the
panels differ in size and in baseline expression, so a raw argmax would favour
whichever panel happens to sit highest.

**Display.** Cluster × state z-matrix, the assignment, and the cell count
falling to each state.

In [ ]:
# STEP 3 — cluster-level state annotation on mg (z-scored argmax)
cat("================================================================\n")
cat("ANNOTATING MICROGLIA STATES (cluster-level, z-scored)\n")
cat("================================================================\n")

CLUSTER_COL <- "mg_cluster"
score_cols  <- paste0(names(gene_sets), "1")
sig_names   <- names(gene_sets)

# ── mean score per subcluster ───────────────────────────────────────────────
cluster_ids <- sort(unique(mg@meta.data[[CLUSTER_COL]]))
cluster_scores <- data.frame(cluster=cluster_ids, row.names=as.character(cluster_ids))
for (col in score_cols)
    cluster_scores[[col]] <- tapply(mg@meta.data[[col]],
                                    mg@meta.data[[CLUSTER_COL]], mean)[as.character(cluster_ids)]

# ── z-score across clusters → assign argmax ─────────────────────────────────
zmat <- scale(as.matrix(cluster_scores[, score_cols])); colnames(zmat) <- sig_names
cluster_scores$assigned_state <- sig_names[apply(zmat, 1, which.max)]
cluster_scores$top_z  <- apply(zmat, 1, max)
cluster_scores$margin <- apply(zmat, 1, function(x){ s<-sort(x,decreasing=TRUE); s[1]-s[2] })
cluster_scores$n_cells <- as.numeric(table(mg@meta.data[[CLUSTER_COL]])[as.character(cluster_ids)])

cat("\nPer-subcluster assignment:\n")
print(cluster_scores[, c("cluster","n_cells","assigned_state","top_z","margin")], digits=3)

low <- cluster_scores$margin < 0.3
if (any(low)) {
    cat("\n⚠ Low-confidence (margin<0.3):\n")
    print(cluster_scores[low, c("cluster","assigned_state","margin")], digits=3)
}

# ── map state to cells ──────────────────────────────────────────────────────
state_map <- setNames(cluster_scores$assigned_state, as.character(cluster_scores$cluster))
mg$microglia_state <- unname(state_map[as.character(mg@meta.data[[CLUSTER_COL]])])

cat("\nState distribution (cells):\n"); print(table(mg$microglia_state))
cat("\nPercent:\n"); print(round(prop.table(table(mg$microglia_state))*100,1))

states_found <- unique(cluster_scores$assigned_state)
missing <- setdiff(sig_names, states_found)
cat("\n✓ States found:", paste(states_found, collapse=", "), "\n")
if (length(missing)>0) cat("⚠ Not detected (no cluster argmax):", paste(missing, collapse=", "), "\n")

---
## 06 · State UMAP

**Why.** The visual check on section 05. States should occupy coherent regions
of `umap.mg`; a state scattered across the embedding means the cluster-level
call is not tracking real structure.

**Display.** `umap.mg` coloured by `microglia_state`, palette and order fixed
by `STATE_COLORS` / `STATE_ORDER` so every figure in every module reads the
same.

In [ ]:
# <<< AXIS-CHECK: hardcoded token(s) ['DAM_like', 'IRM'] remain in this lifted
#     block — replace with X_LAB / AXIS_SPEC$cols[1] / QUAD_COL.
# (2) STATE UMAP on umap.mg — pooled + split by group (Control | AD)
suppressPackageStartupMessages({ library(ggplot2); library(dplyr) })
gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")

emb <- as.data.frame(Embeddings(mg, reduction="umap.mg"))
colnames(emb)[1:2] <- c("UMAP1","UMAP2")
emb$state <- mg$microglia_state
emb$grp2  <- unname(gmap[as.character(mg$Study_Group)])

STATE_COL <- c("Homeostatic"="#B0B7BC",  # grey
               "ARM"        ="#F5A623",  # amber
               "IRM"        ="#2E86C1",  # strong blue
               "Stress"     ="#8E44AD",  # purple
               "DAM_like"   ="#E4002B")  # crimson

ord <- c("Homeostatic","ARM","IRM","Stress","DAM_like")
emb$state <- factor(emb$state, levels=ord)

# robust shared limits
qx <- quantile(emb$UMAP1, c(0.005,0.995)); qy <- quantile(emb$UMAP2, c(0.005,0.995))
xpad <- diff(qx)*0.06; ypad <- diff(qy)*0.06
XLIM <- c(qx[1]-xpad,qx[2]+xpad); YLIM <- c(qy[1]-ypad,qy[2]+ypad)

base_umap <- function(d, ttl, sub) {
    d <- d[sample(nrow(d)), ]
    ggplot(d, aes(UMAP1,UMAP2,color=state)) +
        geom_point(size=2, alpha=1, stroke=0) +
        scale_color_manual(values=STATE_COL, name=NULL,
            guide=guide_legend(override.aes=list(size=2.5,alpha=1))) +
        coord_cartesian(xlim=XLIM, ylim=YLIM, expand=FALSE) +
        labs(title=ttl, subtitle=sub, x="UMAP 1", y="UMAP 2") +
        theme_classic(base_size=10) +
        theme(plot.title=element_text(size=11,face="bold"),
              plot.subtitle=element_text(size=8,color="grey45"),
              axis.text=element_blank(), axis.ticks=element_blank(),
              legend.position="right",
              panel.border=element_rect(color="black",fill=NA,linewidth=0.5),
              aspect.ratio=diff(YLIM)/diff(XLIM))
}

# pooled
cnts <- table(mg$microglia_state)[ord]
p_pool <- base_umap(emb, "Microglia states (all cells)",
    sprintf("%s", paste(sprintf("%s %.1f%%", ord, 100*as.numeric(cnts)/sum(cnts)), collapse=" · ")))
options(repr.plot.width=7.5, repr.plot.height=5); print(p_pool)
save_figure(p_pool, "umap_microglia_states_pooled", width=7.5, height=5)

# split by group
eg <- emb[!is.na(emb$grp2), ]; eg$grp2 <- factor(eg$grp2, levels=c("Control","AD"))
eg <- eg[sample(nrow(eg)), ]
p_split <- base_umap(eg, "Microglia states by group",
                     sprintf("Control vs AD | n=%s", format(nrow(eg),big.mark=","))) +
    facet_wrap(~grp2)
options(repr.plot.width=11, repr.plot.height=5); print(p_split)
save_figure(p_split, "umap_microglia_states_byGroup", width=11, height=5)
cat("\n✓ state UMAPs (pooled + split) done.\n")

---
## 07 · State composition and differential proportion

**Why.** Cell-type proportions are compositional — they sum to one, so a real
increase in one state forces an apparent decrease in the others. Testing raw
proportions with a t-test reports that artifact as a result.

**Test.** Garg et al. 2025 cube-root differential proportion. The cube root
stabilizes the variance of a proportion and makes the residuals roughly
symmetric, which the raw or logit scale does not at proportions near zero.

**Formula.** `cbrt(prop) ~ group + covariates`, donor-level, one model per
state.

**Display.** Composition bars with Garg arrows, then the same stratified by
senescence status. Degenerate fits are suppressed rather than plotted with a
fabricated interval.

In [ ]:
# STATE-LEVEL differential proportion + GLMM  ·  AD vs Old_HC  ·  per-donor
#   states = ARM / DAM_like / Homeostatic / IRM / Stress
#   proportion denominator = donor's TOTAL microglia (within-microglia composition)
suppressPackageStartupMessages({ library(dplyr); library(tidyr); library(lme4); library(knitr); library(IRdisplay) })

DONOR <- "SubID_export_synapse"
GRP   <- "Study_Group"
REF   <- "Old_Healthy_Control"; TEST <- "Old_AD"
COVS  <- c("Sex","Cohort")                 # + Age_dec built below
md <- mg@meta.data

# ── per-donor × state counts (AD + Old_HC only) ─────────────────────────────
sub <- md[md[[GRP]] %in% c(REF,TEST), ]
states <- sort(unique(as.character(sub$microglia_state)))

# donor-level metadata (one row per donor)
dmeta <- sub %>% distinct(.data[[DONOR]], .keep_all=TRUE) %>%
  transmute(donor=.data[[DONOR]], grp=.data[[GRP]], Sex=Sex, Cohort=Cohort,
            Age_dec=as.numeric(Age)/10)

# counts: n cells of each state per donor + donor total microglia
cnt <- sub %>% group_by(donor=.data[[DONOR]], state=microglia_state) %>%
  summarise(n=n(), .groups="drop") %>%
  complete(donor, state, fill=list(n=0))
tot <- sub %>% group_by(donor=.data[[DONOR]]) %>% summarise(total=n(), .groups="drop")

dat <- cnt %>% left_join(tot, by="donor") %>% left_join(dmeta, by="donor") %>%
  mutate(prop = n/total, n_other = total - n,
         grp = relevel(factor(grp), ref=REF))

cat("states:", paste(states, collapse=", "), "\n")
cat("donors:", n_distinct(dat$donor), " (", 
    paste(names(table(dmeta$grp)), table(dmeta$grp), collapse="  "), ")\n\n")

# ── (A) GARG cube-root differential proportion: (prop)^(1/3) ~ grp + covs ────
garg <- lapply(states, function(s){
  d <- dat[dat$state==s, ]
  d$y <- d$prop^(1/3)
  cv <- COVS[sapply(COVS, function(c) length(unique(d[[c]]))>1)]
  f  <- as.formula(paste("y ~ grp + Age_dec", if(length(cv)) paste("+",paste(cv,collapse="+")) else ""))
  m  <- glm(f, data=d, family=gaussian())
  co <- summary(m)$coefficients
  tn <- grep(paste0("grp",TEST), rownames(co), value=TRUE)[1]
  ci <- suppressMessages(confint(m, tn))
  data.frame(state=s, beta=co[tn,"Estimate"], se=co[tn,"Std. Error"],
             lo=ci[1], hi=ci[2], p=co[tn,"Pr(>|t|)"])
}) %>% bind_rows()
garg$padj <- p.adjust(garg$p, "BH")

# ── (B) BINOMIAL GLMM: cbind(n, n_other) ~ grp + Age + Sex + Cohort + (1|donor)
glmm <- lapply(states, function(s){
  d <- dat[dat$state==s, ]
  if (sum(d$n) < 20) return(NULL)
  cv <- COVS[sapply(COVS, function(c) length(unique(d[[c]]))>1)]
  f  <- as.formula(paste("cbind(n, n_other) ~ grp + Age_dec",
                         if(length(cv)) paste("+",paste(cv,collapse="+")) else "", "+ (1|donor)"))
  m  <- tryCatch(glmer(f, data=d, family=binomial,
                       control=glmerControl(optimizer="bobyqa", optCtrl=list(maxfun=1e5))),
                 error=function(e) NULL)
  if (is.null(m)) return(data.frame(state=s, OR=NA, lo=NA, hi=NA, p=NA, sing=NA))
  co <- summary(m)$coefficients
  tn <- grep(paste0("grp",TEST), rownames(co), value=TRUE)[1]
  b  <- co[tn,"Estimate"]; se <- co[tn,"Std. Error"]
  data.frame(state=s, OR=exp(b), lo=exp(b-1.96*se), hi=exp(b+1.96*se),
             p=co[tn,"Pr(>|z|)"], sing=isSingular(m))
}) %>% bind_rows()
glmm$padj <- p.adjust(glmm$p, "BH")

# ── display ──────────────────────────────────────────────────────────────────
fp <- function(x) ifelse(is.na(x),"—", ifelse(x<0.001,"<0.001", sprintf("%.3f",x)))
GA <- garg %>% transmute(State=state, `cube-root β`=round(beta,4),
        `95% CI`=sprintf("[%.3f, %.3f]",lo,hi), p=fp(p),
        FDR=paste0(fp(padj), ifelse(padj<0.05," *",""))) %>% arrange(desc(garg$beta))
GL <- glmm %>% transmute(State=state, OR=round(OR,3),
        `95% CI`=sprintf("[%.2f, %.2f]",lo,hi), p=fp(p),
        FDR=paste0(fp(padj), ifelse(padj<0.05," *","")),
        flag=ifelse(sing %in% TRUE,"singular","")) %>% arrange(desc(glmm$OR))

show_tbl <- function(df,title,note=""){
  display_html(sprintf("<h4 style='margin:12px 0 2px;font-family:sans-serif'>%s</h4><p style='margin:0 0 6px;color:#666;font-size:12px;font-family:sans-serif'>%s</p>",title,note))
  display_html(as.character(kable(df, format="html", align="r", row.names=FALSE,
    table.attr="style='font-size:13px;border-collapse:collapse;font-family:sans-serif'")))
}
show_tbl(GA,"State · Garg differential proportion (AD vs Old HC)",
  "(prop)^(1/3) ~ Study_Group + Age + Sex + Cohort · prop = state share of donor's microglia · FDR-BH · * &lt; 0.05")
show_tbl(GL,"State · Binomial GLMM (AD vs Old HC)",
  "cbind(n, n_other) ~ Study_Group + Age + Sex + Cohort + (1|donor) · OR = AD odds of being this state · * FDR &lt; 0.05")

In [ ]:
# (1) Garg cube-root proportion test, STRATIFIED by senescence status
#     per state, within nSnC and within SnC separately:  AD vs Old HC
#     leaves `garg_snc` in the session for the plotting cell
suppressPackageStartupMessages({ library(dplyr); library(tidyr) })

DONOR <- "SubID_export_synapse"; GRP <- "Study_Group"; SEN <- "is_senescent"
GROUP_ORDER <- c("Old_Healthy_Control","Old_AD")
MIN_CELLS   <- 10        # min microglia of that status per donor to include the donor×stratum

md <- mg@meta.data
md <- md[md[[GRP]] %in% GROUP_ORDER, ]
md <- data.frame(
    donor = as.character(md[[DONOR]]),
    grp   = factor(as.character(md[[GRP]]), levels = GROUP_ORDER),
    state = factor(as.character(md$microglia_state), levels = STATE_ORDER),
    SnC   = ifelse(as.integer(md[[SEN]]) == 1, "SnC", "nSnC"),
    stringsAsFactors = FALSE) %>%
    filter(!is.na(state))

# per-donor × status: state counts, denominator = that donor's cells of that status
dd <- md %>%
    count(grp, donor, SnC, state, name = "n") %>%
    group_by(donor, SnC) %>% mutate(n_tot = sum(n)) %>% ungroup() %>%
    filter(n_tot >= MIN_CELLS) %>%
    complete(nesting(grp, donor, SnC, n_tot), state, fill = list(n = 0)) %>%
    mutate(prop = n / n_tot)

# Garg cube-root proportion model, one fit per (SnC stratum × state)
garg_one <- function(d) {
    d$y <- d$prop^(1/3)
    m   <- lm(y ~ grp, data = d)
    co  <- summary(m)$coefficients
    if (!"grpOld_AD" %in% rownames(co))
        return(data.frame(beta=NA, se=NA, p=NA, n_hc=NA, n_ad=NA))
    data.frame(beta = co["grpOld_AD","Estimate"],
               se   = co["grpOld_AD","Std. Error"],
               p    = co["grpOld_AD","Pr(>|t|)"],
               n_hc = sum(d$grp == "Old_Healthy_Control"),
               n_ad = sum(d$grp == "Old_AD"))
}

garg_snc <- dd %>%
    group_by(SnC, state) %>% group_modify(~ garg_one(.x)) %>% ungroup() %>%
    group_by(SnC) %>% mutate(padj = p.adjust(p, "BH")) %>% ungroup() %>%
    mutate(dir = sign(beta), sig = padj < 0.05)

# attach raw group-mean % for readability
gmean <- dd %>% group_by(SnC, state, grp) %>%
    summarise(mpct = 100*mean(prop), .groups="drop") %>%
    pivot_wider(names_from = grp, values_from = mpct)
garg_snc <- garg_snc %>% left_join(gmean, by = c("SnC","state"))

cat("═══ Garg cube-root: AD vs Old HC, within each senescence stratum ═══\n")
cat(sprintf("(min %d microglia of that status per donor)\n\n", MIN_CELLS))
print(garg_snc %>%
      mutate(across(c(beta,se,p,padj), ~signif(.x,3)),
             across(where(is.numeric) & !c(beta,se,p,padj), ~round(.x,1))) %>%
      arrange(SnC, state) %>% as.data.frame())
cat(sprintf("\ndonors surviving MIN_CELLS per stratum: nSnC=%d, SnC=%d\n",
            n_distinct(dd$donor[dd$SnC=="nSnC"]), n_distinct(dd$donor[dd$SnC=="SnC"])))

In [ ]:
# (2) State composition by senescence status, with Garg arrows from garg_snc
suppressPackageStartupMessages({ library(dplyr); library(ggplot2) })
stopifnot(exists("garg_snc"), exists("comp_snc"))

comp_snc$lbl <- ifelse(comp_snc$pct >= 4, sprintf("%.0f%%", comp_snc$pct), "")

# arrowheads: significant AD shift within each stratum, centered on the AD segment
ad_seg <- comp_snc %>%
    filter(Group == "Old_AD") %>%
    group_by(SnC) %>% arrange(match(State, rev(STATE_ORDER)), .by_group=TRUE) %>%
    mutate(cum = cumsum(pct), center = cum - pct/2) %>% ungroup()

arr <- garg_snc %>% filter(sig %in% TRUE) %>%
    transmute(SnC, State = state, dir) %>%
    left_join(ad_seg %>% select(SnC, State, center), by = c("SnC","State")) %>%
    mutate(Group = factor("Old_AD", levels = levels(comp_snc$Group)),
           SnC   = factor(SnC,      levels = levels(comp_snc$SnC)))

p_snc <- ggplot(comp_snc, aes(x=pct, y=Group, fill=State)) +
    geom_col(aes(alpha=SnC, linetype=SnC), width=0.72, color="grey30", linewidth=0.4) +
    geom_text(aes(label=lbl, color=SnC), position=position_stack(vjust=0.5),
              size=2.5, show.legend=FALSE) +
    geom_point(data=arr, aes(x=center, y=Group, shape=ifelse(dir>0,24,25)),
               inherit.aes=FALSE, fill="white", color="grey20", size=2.4, stroke=0.7) +
    scale_shape_identity() +
    facet_wrap(~SnC, nrow=1,
               labeller=as_labeller(c(nSnC="Non-senescent", SnC="Senescent"))) +
    scale_fill_manual(values=STATE_COLORS, breaks=STATE_ORDER) +
    scale_alpha_manual(values=c(nSnC=1, SnC=0.55), guide="none") +
    scale_linetype_manual(values=c(nSnC="solid", SnC="22"), guide="none") +
    scale_color_manual(values=c(nSnC="white", SnC="grey20"), guide="none") +
    scale_y_discrete(limits=rev(levels(comp_snc$Group)), labels=GROUP_LAB) +
    scale_x_continuous(expand=expansion(mult=c(0,0.02))) +
    coord_cartesian(xlim=c(0,100)) +
    labs(x="% of microglia", y=NULL, fill="State",
         title="Microglial state composition: non-senescent vs senescent",
         subtitle="AD vs Old HC within each stratum · Garg cube-root · \u25b2/\u25bc FDR<0.05") +
    theme_classic(base_size=10) +
    theme(plot.subtitle=element_text(size=8, color="#666666"),
          strip.background=element_blank(), strip.text=element_text(face="bold", size=10),
          panel.border=element_rect(color="#333333", fill=NA, linewidth=0.5), axis.line=element_blank(),
          legend.key.size=unit(0.4,"cm"))
save_figure(p_snc, "mg_state_composition_by_snc", width=9, height=3.6)
options(repr.plot.width=9, repr.plot.height=3.6); print(p_snc)

In [ ]:
# <<< AXIS-CHECK: hardcoded token(s) ['DAM_like', 'IRM'] remain in this lifted
#     block — replace with X_LAB / AXIS_SPEC$cols[1] / QUAD_COL.
# STATE COMPOSITION 3-PANEL — AD progression (microglia)
#   A · State proportion        (prop_res; Garg cube-root)
#   B · Senescent burden        (burden_ctadj; GLMM adjusted for state proportion)
#   C · Senescence susceptibility (susc_res; per-cell GLMM)
#   Significance: ▲/▼ on A & B, * on C — FDR<0.05, degenerate fits (IRM) suppressed
suppressPackageStartupMessages({ library(dplyr); library(tidyr); library(ggplot2); library(patchwork) })

# ── config ──────────────────────────────────────────────────────────────────
DONOR <- "SubID_export_synapse"; GRP <- "Study_Group"; SEN <- "is_senescent"
STATE_COLORS <- c(Homeostatic="#7F8C8D", ARM="#E67E22", IRM="#2980B9",
                  Stress="#8E44AD", DAM_like="#C0392B")
STATE_ORDER  <- c("Homeostatic","IRM","ARM","Stress","DAM_like")
GROUP_ORDER  <- c("Old_Healthy_Control","Old_AD") #"Young_Healthy_Control",
GLAB <- c(Young_Healthy_Control="Young HC", Old_Healthy_Control="Old HC", Old_AD="AD")
AD_LAB <- "AD"                                  # must match GLAB["Old_AD"]
GLAB_ORDER <- unname(GLAB[GROUP_ORDER])

# ── degenerate-fit guard (impossibly tight CI or p exactly 0 → unreliable) ───
flag_degenerate <- function(df){
  df$ok <- !(((df$hi - df$lo)/df$OR) < 0.05 | df$p == 0)
  df
}
burden_ctadj <- flag_degenerate(burden_ctadj)
susc_res     <- flag_degenerate(susc_res)

# ── data: per-donor metrics → group means ────────────────────────────────────
md <- mg@meta.data
md <- md[md[[GRP]] %in% GROUP_ORDER, ]
md$state <- factor(as.character(md$microglia_state), levels=STATE_ORDER)
md$grp   <- factor(GLAB[as.character(md[[GRP]])], levels=rev(GLAB_ORDER))
md$snc   <- as.integer(md[[SEN]]); md$donor <- md[[DONOR]]

dd <- md %>% group_by(grp, donor, state) %>%
  summarise(n=n(), n_snc=sum(snc), .groups="drop")
dtot <- md %>% group_by(donor) %>% summarise(mg_tot=n(), .groups="drop")
dd <- dd %>% left_join(dtot, by="donor") %>%
  mutate(prop=n/mg_tot, burden=n_snc/mg_tot, senfrac=ifelse(n>=10, n_snc/n, NA))
gm <- dd %>% group_by(grp, state) %>%
  summarise(prop=mean(prop,na.rm=TRUE), burden=mean(burden,na.rm=TRUE),
            senfrac=mean(senfrac,na.rm=TRUE), .groups="drop") %>%
  complete(grp, state, fill=list(prop=0,burden=0,senfrac=0))
gm <- gm %>% group_by(grp) %>% mutate(prop_norm=100*prop/sum(prop)) %>% ungroup()
gm$burden_pct <- 100*gm$burden

# ── arrow positions on AD row (significant + non-degenerate only) ────────────
arrow_df <- function(sig, valuecol, gmcol){
  ad <- gm %>% filter(grp==AD_LAB) %>% arrange(match(state, rev(STATE_ORDER)))
  ad$cum <- cumsum(ad[[gmcol]]); ad$center <- ad$cum - ad[[gmcol]]/2
  s <- sig %>% mutate(dir = if(valuecol=="beta") sign(beta) else sign(log(OR)),
                      keep = padj<0.05 & (if("ok" %in% names(sig)) ok else TRUE)) %>%
       select(state, dir, keep)
  ad %>% left_join(s, by="state") %>% filter(keep %in% TRUE) %>%
    mutate(grp = factor(AD_LAB, levels=rev(GLAB_ORDER)))
}
ap <- arrow_df(prop_res,     "beta", "prop_norm")
ab <- arrow_df(burden_ctadj, "OR",   "burden_pct")

# ── theme ─────────────────────────────────────────────────────────────────────
base_theme <- theme_classic(base_size=9) +
  theme(plot.title=element_text(face="bold", size=10, hjust=0.5),
        axis.title.x=element_text(size=8.5), axis.text=element_text(size=8),
        axis.ticks.y=element_blank(), legend.position="none", plot.margin=margin(4,6,4,4))

# ── A · stacked proportion + arrows ──────────────────────────────────────────
gA <- ggplot(gm, aes(prop_norm, grp, fill=state)) +
  geom_col(width=0.66, colour="white", linewidth=0.3) +
  geom_point(data=ap, aes(x=center, y=grp, shape=ifelse(dir>0,24,25)),
             inherit.aes=FALSE, fill="white", colour="grey20", size=2.6, stroke=0.7) +
  scale_shape_identity() +
  scale_fill_manual(values=STATE_COLORS, breaks=STATE_ORDER) +
  scale_x_continuous(expand=expansion(mult=c(0,0.02))) +
  labs(title="A · State proportion", x="% of microglia", y=NULL) + base_theme

# ── B · stacked burden (ct_prop-adjusted) + totals + arrows ──────────────────
tot_b <- gm %>% group_by(grp) %>% summarise(tot=sum(burden_pct), .groups="drop")
gB <- ggplot(gm, aes(burden_pct, grp, fill=state)) +
  geom_col(width=0.66, colour="white", linewidth=0.3) +
  geom_text(data=tot_b, aes(x=tot, y=grp, label=sprintf("%.2f%%", tot)),
            inherit.aes=FALSE, hjust=-0.1, size=2.8, fontface="bold", colour="grey20") +
  geom_point(data=ab, aes(x=center, y=grp, shape=ifelse(dir>0,24,25)),
             inherit.aes=FALSE, fill="white", colour="grey20", size=2.6, stroke=0.7) +
  scale_shape_identity() +
  scale_fill_manual(values=STATE_COLORS, breaks=STATE_ORDER) +
  scale_x_continuous(expand=expansion(mult=c(0,0.14))) +
  labs(title="B · Senescent burden", x="Senescent microglia (% of all microglia)", y=NULL) +
  base_theme

# ── C · grouped susceptibility + significance stars ──────────────────────────
sc  <- susc_res %>% mutate(sig = padj<0.05 & ok) %>% select(state, sig)
gmc <- gm %>% left_join(sc, by="state") %>% filter(grp==AD_LAB)
gC <- ggplot(gm, aes(100*senfrac, grp, fill=state)) +
  geom_col(width=0.72, position=position_dodge(width=0.78), colour="white", linewidth=0.2) +
  geom_text(data=gmc %>% filter(sig %in% TRUE),
            aes(x=100*senfrac, y=grp, label="*"), inherit.aes=FALSE,
            position=position_dodge(width=0.78), hjust=-0.4, size=4, colour="grey20") +
  scale_fill_manual(values=STATE_COLORS, breaks=STATE_ORDER) +
  scale_x_continuous(expand=expansion(mult=c(0,0.06))) +
  labs(title="C · Senescence susceptibility", x="Senescent fraction within state (%)", y=NULL) +
  base_theme

# ── assemble ──────────────────────────────────────────────────────────────────
final <- (gA | gB | gC) + plot_layout(widths=c(1,1,1.15), guides="collect") &
  theme(legend.position="bottom", legend.title=element_blank(),
        legend.text=element_text(size=8))

final <- final + plot_annotation(
  title="Microglial state composition · senescent burden · susceptibility — AD progression",
  caption=paste0("Young HC = aging reference; statistical test is AD vs Old HC. ",
    "\u25b2/\u25bc (A,B) and * (C) = FDR<0.05. Proportion via Garg cube-root; ",
    "burden via GLMM adjusted for state proportion; susceptibility via per-cell GLMM. ",
    "IRM excluded from B/C significance (degenerate model fit); ",
    "Stress susceptibility based on few cells (interpret with caution)."),
  theme=theme(plot.title=element_text(face="bold", size=10.5, hjust=0.5),
              plot.caption=element_text(size=6, colour="grey45", hjust=0.5)))

options(repr.plot.width=12.5, repr.plot.height=3.6)
print(final)
ggsave("state_composition_3panel_disease.pdf", final, width=12.5, height=3.6, device=cairo_pdf)
ggsave("state_composition_3panel_disease.png", final, width=12.5, height=3.6, dpi=300, bg="white")
ggsave("state_composition_3panel_disease.svg", final,width = 12.5, height = 3.6, device = svglite::svglite)
cat("\n\u2713 state composition 3-panel saved (AD label; burden ct_prop-adjusted; IRM guarded)\n")

---
## 08 · State-level burden GLMM

**Why.** Composition says how many cells are in a state. This asks whether the
cells *in* that state are more likely to be senescent — the state-resolved
version of the susceptibility question from module 07.

**Test.** Binomial GLMM with a donor random intercept, adjusted for the state's
own proportion (`ct_prop`) so that a state cannot appear enriched purely by
having grown.

**Formula.** `is_senescent ~ group + ct_prop + covariates + (1 | Donor)`

**Display.** Per-state odds ratios, used as the arrows on the composition panel.

In [ ]:
# ct_prop-ADJUSTED BURDEN GLMM — per microglia state (for Panel B arrows)
#   burden = senescent cells of state S as fraction of ALL microglia
#   adjusts for ct_prop_z (state's donor-level proportion) — appropriate for burden
suppressPackageStartupMessages({ library(lme4); library(dplyr) })
DONOR<-"SubID_export_synapse"; GRP<-"Study_Group"
STATE_ORDER <- c("Homeostatic","IRM","ARM","Stress","DAM_like")
REF <- "Old_Healthy_Control"; TESTG <- "Old_AD"   # AD vs Old HC (matches the figure's test)

md <- mg@meta.data
md <- md[md[[GRP]] %in% c(REF,TESTG), ]          # AD vs Old HC only (drop Young for the test)
md$donor <- md[[DONOR]]; md$grp2 <- relevel(factor(md[[GRP]]), ref=REF)
md$log10_UMI <- log10(md$nCount_RNA + 1)
md$Sex <- factor(md$Sex); md$Cohort <- factor(make.names(as.character(md$Cohort)))

# donor-level proportion of each state
dtot <- md %>% count(donor, name="n_tot")
ctp  <- md %>% count(donor, microglia_state, name="n_ct") %>%
        left_join(dtot, by="donor") %>% mutate(ct_prop=n_ct/n_tot)

ctrl <- glmerControl(optimizer="bobyqa", optCtrl=list(maxfun=1e5))
burden_ctadj <- lapply(STATE_ORDER, function(S){
  d <- md
  d$is_burden <- as.integer(d$is_senescent==1 & d$microglia_state==S)
  # attach this state's ct_prop per donor
  pp <- ctp %>% filter(microglia_state==S) %>% select(donor, ct_prop)
  d <- d %>% left_join(pp, by="donor")
  d$ct_prop[is.na(d$ct_prop)] <- 0
  d$ct_prop_z <- as.numeric(scale(d$ct_prop))
  d$Cohort <- droplevels(d$Cohort)
  cov <- if (nlevels(d$Cohort)>1) "Sex + Cohort + log10_UMI" else "Sex + log10_UMI"
  f <- as.formula(paste("is_burden ~ grp2 + ct_prop_z +", cov, "+ (1|donor)"))
  m <- tryCatch(glmer(f, data=d, family=binomial, control=ctrl, nAGQ=1), error=function(e) NULL)
  if (is.null(m)) return(data.frame(state=S, OR=NA, lo=NA, hi=NA, p=NA, sing=NA))
  s <- summary(m)$coefficients; idx <- grep("^grp2", rownames(s))[1]
  b<-s[idx,"Estimate"]; se<-s[idx,"Std. Error"]; p<-s[idx,"Pr(>|z|)"]
  data.frame(state=S, OR=exp(b), lo=exp(b-1.96*se), hi=exp(b+1.96*se), p=p,
             sing=isSingular(m))
}) %>% bind_rows()
burden_ctadj$padj <- p.adjust(burden_ctadj$p, "BH")
cat("ct_prop-ADJUSTED BURDEN (AD vs Old HC):\n"); print(burden_ctadj)

---
## 09 · Subtypes as a share of all cells

**Why.** Reframes composition against the whole tissue rather than against the
cell type. A state can hold a stable share of microglia while microglia
themselves change as a fraction of the brain — this section makes that visible,
and it is the same burden-versus-proportion distinction drawn in module 07.

In [ ]:
# MICROGLIAL SUBTYPES AS % OF ALL CELLS  —  AD vs age-matched control
#   Q: does the microglial compartment itself change, not just its internal mix?
#   inherits STATE_ORDER / STATE_COLORS / GROUP_ORDER / GROUP_LAB from earlier
suppressPackageStartupMessages({ library(dplyr); library(tidyr); library(ggplot2) })

FULL_OBJ  <- obj      # <<< SET THIS: object containing ALL cell types
mg_obj    <- mg
DONOR_COL <- "Donor"; GROUP_COL <- "Study_Group"; STATE_COL <- "microglia_state"

# ─── denominator: total nuclei per donor (from the FULL object) ─────────────
fm <- FULL_OBJ@meta.data
stopifnot(DONOR_COL %in% colnames(fm))
donor_tot <- fm %>%
    transmute(Donor = as.character(.data[[DONOR_COL]])) %>%
    count(Donor, name = "n_all")

# fallback if the full object isn't loaded — supply a Donor,n_all table instead:
# donor_tot <- read.csv("donor_total_cells.csv", stringsAsFactors=FALSE)

# ─── numerator: microglia per donor x state ────────────────────────────────
mm <- mg_obj@meta.data
mm <- mm[mm[[GROUP_COL]] %in% GROUP_ORDER, ]
state_n <- data.frame(
        Donor = as.character(mm[[DONOR_COL]]),
        Group = as.character(mm[[GROUP_COL]]),
        State = as.character(mm[[STATE_COL]]),
        stringsAsFactors = FALSE) %>%
    filter(!is.na(State), State %in% STATE_ORDER, !is.na(Donor)) %>%
    count(Donor, Group, State, name = "n_state")

comp_all <- state_n %>%
    complete(nesting(Donor, Group), State, fill = list(n_state = 0)) %>%
    left_join(donor_tot, by = "Donor")

miss <- comp_all %>% filter(is.na(n_all)) %>% distinct(Donor)
if (nrow(miss)) { cat("!! donors absent from FULL_OBJ (dropped):\n"); print(miss) }

comp_all <- comp_all %>%
    filter(!is.na(n_all)) %>%
    mutate(pct_all = 100 * n_state / n_all,
           State = factor(State, levels = STATE_ORDER),
           Group = factor(Group, levels = GROUP_ORDER))

# total microglia as % of all cells, per donor
mg_frac <- comp_all %>%
    group_by(Donor, Group) %>%
    summarise(pct_all = sum(pct_all), .groups = "drop") %>%
    mutate(State = factor("ALL microglia", levels = "ALL microglia"))

# ─── group summary + AD vs Old HC test (per state, BH across states) ───────
summarise_prop <- function(d) {
    d %>% group_by(State, Group) %>%
        summarise(mean = mean(pct_all), sd = sd(pct_all),
                  median = median(pct_all), n_donor = n(), .groups = "drop")
}
test_prop <- function(d) {
    d %>% group_by(State) %>%
        summarise(
            hc      = mean(pct_all[Group == GROUP_ORDER[1]]),
            ad      = mean(pct_all[Group == GROUP_ORDER[2]]),
            diff_pp = ad - hc,
            ratio   = ad / hc,
            p_wilcox = tryCatch(
                wilcox.test(pct_all ~ Group, exact = FALSE)$p.value, error = function(e) NA_real_),
            # cube-root transform = same variance-stabilising step as the Garg proportion models
            p_cbrt   = tryCatch(
                summary(lm(pct_all^(1/3) ~ Group))$coefficients[2, 4], error = function(e) NA_real_),
            .groups = "drop") %>%
        mutate(fdr_wilcox = p.adjust(p_wilcox, "BH"),
               fdr_cbrt   = p.adjust(p_cbrt,   "BH"))
}

cat("\n═══ Microglial subtypes as % of ALL cells ═══\n\n")
print(summarise_prop(comp_all) %>%
        mutate(across(where(is.numeric), ~round(.x, 3))) %>% as.data.frame())
cat("\n── total microglia (% of all cells) ──\n")
print(summarise_prop(mg_frac) %>%
        mutate(across(where(is.numeric), ~round(.x, 3))) %>% as.data.frame())
cat("\n── AD vs", GROUP_LAB[[GROUP_ORDER[1]]], "──\n")
print(bind_rows(test_prop(comp_all), test_prop(mg_frac)) %>%
        mutate(across(where(is.numeric), ~signif(.x, 3))) %>% as.data.frame())

# ─── (A) stacked bar: mean % of all cells (NOT normalised to 100) ──────────
grp_all <- comp_all %>% group_by(Group, State) %>%
    summarise(mean_pct = mean(pct_all), .groups = "drop")
tot_lab <- grp_all %>% group_by(Group) %>%
    summarise(tot = sum(mean_pct), .groups = "drop")

p_all <- ggplot(grp_all, aes(Group, mean_pct, fill = State)) +
    geom_col(width = 0.66, color = "white", linewidth = 0.3) +
    geom_text(data = tot_lab, aes(Group, tot, label = sprintf("%.2f%%", tot)),
              inherit.aes = FALSE, vjust = -0.5, size = 3, fontface = "bold", colour = "grey20") +
    scale_fill_manual(values = STATE_COLORS, breaks = STATE_ORDER) +
    scale_x_discrete(labels = GROUP_LAB) +
    scale_y_continuous(expand = expansion(mult = c(0, 0.12))) +
    labs(x = NULL, y = "Mean % of all cells", fill = "State",
         title = "Microglial subtypes as a fraction of all cells") +
    theme_classic(base_size = 10) +
    theme(panel.border = element_rect(color = "#333333", fill = NA, linewidth = 0.5),
          axis.line = element_blank(), legend.key.size = unit(0.4, "cm"))

# ─── (B) per-donor distributions, one facet per state + total ─────────────
box_df <- bind_rows(comp_all %>% select(Donor, Group, State, pct_all),
                    mg_frac  %>% select(Donor, Group, State, pct_all)) %>%
    mutate(State = factor(as.character(State),
                          levels = c("ALL microglia", STATE_ORDER)))
box_pal <- c("ALL microglia" = "#34495E", STATE_COLORS)

p_box <- ggplot(box_df, aes(Group, pct_all)) +
    geom_boxplot(fill = "white", colour = "black", outlier.shape = NA,
                 width = 0.55, linewidth = 0.4) +
    geom_jitter(aes(colour = State), width = 0.15, size = 0.9, alpha = 0.7,
                show.legend = FALSE) +
    facet_wrap(~State, scales = "free_y", nrow = 1) +
    scale_colour_manual(values = box_pal) +
    scale_x_discrete(labels = GROUP_LAB) +
    labs(x = NULL, y = "% of all cells",
         title = "Per-donor microglial subtype abundance (fraction of all cells)") +
    theme_classic(base_size = 10) +
    theme(panel.border = element_rect(color = "#333333", fill = NA, linewidth = 0.5),
          axis.line = element_blank(),
          strip.background = element_blank(),
          strip.text = element_text(face = "bold", size = 8))

save_figure(p_all, "mg_subtype_pct_of_all_cells",       width = 5,  height = 4.5)
save_figure(p_box, "mg_subtype_pct_of_all_cells_donor", width = 11, height = 3.2)
options(repr.plot.width = 5,  repr.plot.height = 4.5); print(p_all)
options(repr.plot.width = 11, repr.plot.height = 3.2); print(p_box)
cat("\n\u2713 subtype fraction-of-all-cells figures saved\n")

---
## 10 · Is the selected state higher in disease?

**Why.** The direct question, asked three ways because the three can disagree:
as a percentage of the cell type, as a percentage of all cells, and as a raw
count per donor. Agreement across the three is what makes the claim safe;
disagreement localises it to an abundance shift somewhere in the denominator.

**Display.** Three panels, one per framing, for the state named by `AXIS`.

In [ ]:
# IS DAM HIGHER IN AD vs AGE-MATCHED CONTROL?
#   three framings: % of microglia · % of all cells · count model w/ offset
suppressPackageStartupMessages({ library(dplyr); library(tidyr); library(ggplot2) })

TARGET_STATE <- AXIS
FULL_OBJ     <- obj        # <<< object with ALL cell types (for n_all)
COVARS       <- character(0)      # e.g. c("Sex","Cohort") if present in mg meta

# ─── per-donor counts ──────────────────────────────────────────────────────
mm <- mg@meta.data
mm <- mm[mm[[GROUP_COL]] %in% GROUP_ORDER, ]
mm$Donor <- as.character(mm[[DONOR_COL]])
mm$Group <- as.character(mm[[GROUP_COL]])
mm$State <- as.character(mm[[STATE_COL]])

cov_df <- if (length(COVARS)) mm %>% group_by(Donor) %>%
              summarise(across(all_of(COVARS), ~ .x[1]), .groups="drop") else NULL

donor_df <- mm %>%
    group_by(Donor, Group) %>%
    summarise(n_mg  = n(),
              n_dam = sum(State == TARGET_STATE, na.rm = TRUE), .groups = "drop")

donor_tot <- FULL_OBJ@meta.data %>%
    transmute(Donor = as.character(.data[[DONOR_COL]])) %>%
    count(Donor, name = "n_all")

donor_df <- donor_df %>%
    left_join(donor_tot, by = "Donor") %>%
    filter(!is.na(n_all)) %>%
    mutate(pct_mg  = 100 * n_dam / n_mg,
           pct_all = 100 * n_dam / n_all,
           Group   = factor(Group, levels = GROUP_ORDER))
if (!is.null(cov_df)) donor_df <- donor_df %>% left_join(cov_df, by = "Donor")

cat(sprintf("\n%s — donors: %s\n", TARGET_STATE,
    paste(sprintf("%s n=%d", GROUP_LAB[levels(donor_df$Group)],
                  as.integer(table(donor_df$Group))), collapse = " | ")))
cat("\nraw totals per group:\n")
print(donor_df %>% group_by(Group) %>%
      summarise(dam_cells = sum(n_dam), microglia = sum(n_mg),
                all_cells = sum(n_all), .groups="drop") %>% as.data.frame())

cat("\nper-donor summary:\n")
print(donor_df %>% group_by(Group) %>%
      summarise(pct_mg_mean = mean(pct_mg),   pct_mg_med  = median(pct_mg),
                pct_all_mean = mean(pct_all), pct_all_med = median(pct_all),
                .groups="drop") %>%
      mutate(across(where(is.numeric), ~round(.x, 3))) %>% as.data.frame())

# ─── tests ─────────────────────────────────────────────────────────────────
rhs <- paste(c("Group", COVARS), collapse = " + ")

res_prop <- lapply(c("pct_mg","pct_all"), function(v) {
    f_lin <- as.formula(sprintf("%s^(1/3) ~ %s", v, rhs))
    data.frame(
        outcome  = v,
        hc       = mean(donor_df[[v]][donor_df$Group == GROUP_ORDER[1]]),
        ad       = mean(donor_df[[v]][donor_df$Group == GROUP_ORDER[2]]),
        p_wilcox = wilcox.test(donor_df[[v]] ~ donor_df$Group, exact = FALSE)$p.value,
        p_cbrt   = summary(lm(f_lin, data = donor_df))$coefficients[2, 4],
        stringsAsFactors = FALSE)
}) %>% bind_rows() %>% mutate(diff_pp = ad - hc, ratio = ad / hc)

cat("\n── proportion tests ──\n")
print(res_prop %>% mutate(across(where(is.numeric), ~signif(.x, 3))) %>% as.data.frame())

# negative-binomial count model: DAM counts, offset for donor depth
nb_fit <- function(offset_col) {
    f <- as.formula(sprintf("n_dam ~ %s + offset(log(%s))", rhs, offset_col))
    m <- try(MASS::glm.nb(f, data = donor_df), silent = TRUE)
    if (inherits(m, "try-error")) return(data.frame(offset = offset_col, IRR = NA, p = NA))
    co <- summary(m)$coefficients[2, ]
    data.frame(offset = offset_col, IRR = exp(co[1]),
               lo = exp(co[1] - 1.96*co[2]), hi = exp(co[1] + 1.96*co[2]), p = co[4])
}
cat("\n── negative-binomial count model (AD vs HC; IRR>1 = more DAM in AD) ──\n")
print(bind_rows(nb_fit("n_mg"), nb_fit("n_all")) %>%
      mutate(across(where(is.numeric), ~signif(.x, 3))) %>% as.data.frame())

# ─── plot ──────────────────────────────────────────────────────────────────
plot_df <- donor_df %>%
    select(Donor, Group, pct_mg, pct_all) %>%
    pivot_longer(c(pct_mg, pct_all), names_to = "denom", values_to = "pct") %>%
    mutate(denom = factor(denom, levels = c("pct_mg","pct_all"),
                          labels = c("% of microglia","% of all cells")))

p_dam <- ggplot(plot_df, aes(Group, pct)) +
    geom_boxplot(fill = "white", colour = "black", outlier.shape = NA,
                 width = 0.55, linewidth = 0.4) +
    geom_jitter(width = 0.15, size = 1.1, alpha = 0.75,
                colour = unname(STATE_COLORS[TARGET_STATE])) +
    facet_wrap(~denom, scales = "free_y") +
    scale_x_discrete(labels = GROUP_LAB) +
    labs(x = NULL, y = NULL, title = paste0(TARGET_STATE, " abundance: AD vs age-matched control")) +
    theme_classic(base_size = 10) +
    theme(panel.border = element_rect(color = "#333333", fill = NA, linewidth = 0.5),
          axis.line = element_blank(), strip.background = element_blank(),
          strip.text = element_text(face = "bold", size = 9))

save_figure(p_dam, "dam_abundance_ad_vs_hc", width = 6, height = 3.4)
options(repr.plot.width = 6, repr.plot.height = 3.4); print(p_dam)